In [1]:
from ultralytics import YOLO
import torch

yolo = YOLO("yolov8s.pt")
model = yolo.model      

In [2]:
import cv2
import torch
from torch.utils.data import Dataset
import numpy as np

def load_yolo_label(label_path):
    boxes = []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, x, y, w, h = map(float, line.split())
            boxes.append([cls, x, y, w, h])
    return np.array(boxes)

class YOLODataset(Dataset):
    def __init__(self,path,img_size=640):
        self.image_path=os.path.join(path,"images")
        self.label_path=os.path.join(path,"labels")
       
        
        self.image_files = sorted(os.listdir(self.image_path))
        # self.label_files = sorted(os.listdir(self.label_path))

        self.img_size = img_size

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        label_name = img_name.replace(".png", ".txt")
        
        img = cv2.imread(os.path.join(self.image_path, img_name))
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img[:, :, ::-1] / 255.0
        img = torch.tensor(img).permute(2, 0, 1).float()

        cur_label_path=os.path.join(self.label_path,label_name)
        label=load_yolo_label(cur_label_path)
        label=np.array(label)

        

        new_label = torch.from_numpy(label).float()

        return img, new_label

In [3]:
from torch.utils.data import DataLoader
import os
path=r"D:\Ml Dl\Project\LunaCraterNet\artifacts\data_ingestion\dataset\LU3M6TGT_yolo_format\train"
dataset = YOLODataset(path,)

def yolo_collate_fn(batch):
    images, targets = zip(*batch)

    images = torch.stack(images, dim=0)
    return images, targets


loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=yolo_collate_fn
)


In [4]:
ip,op= next(iter(loader))
ip.shape

torch.Size([4, 3, 640, 640])

In [6]:
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [12]:
# op=op.to(device)
for item in op:
    item=item.to(device)
    

In [9]:
ip=ip.to(device)

In [10]:
op[0].shape

torch.Size([88, 5])

In [ ]:
preds = model(ip)
preds[0].shape

In [ ]:
len(preds)

In [ ]:
model.loss(preds, op)